# env_functions

- Este notebook armazena os métodos usados ao longo de todo o pipeline MUNIC

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
def upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description):
    """Upsert de um Spark DataFrame em uma tabela Delta particionada por ano_edicao.

    Adiciona 'ano_edicao' à origem automaticamente, se ainda não estiver presente.
    Trata a criação da tabela, a migração de tabelas não particionadas, a evolução
    de schema via ALTER TABLE e o upsert via MERGE com chaves compostas.

    Parameters
    ----------
    df_source    : Spark DataFrame a ser gravado (upsert)
    table_name   : nome totalmente qualificado da tabela de destino
    merge_keys   : lista de colunas para a condição do MERGE
    ano_edicao   : ano da edição da pesquisa (int)
    description  : string gravada no userMetadata da operação Delta
    """
    # Garante que ano_edicao exista na origem; posiciona-a em segundo (após a chave da entidade)
    source_cols = df_source.columns
    if "ano_edicao" not in source_cols:
        first_col = source_cols[0]
        rest_cols = source_cols[1:]
        df_source = df_source.withColumns({"ano_edicao": F.lit(ano_edicao)})
        df_source = df_source.select(first_col, "ano_edicao", *rest_cols)
        source_cols = [first_col, "ano_edicao", *rest_cols]

    # Ex.: "target.ano_edicao = source.ano_edicao AND target.cod_munic = source.cod_munic"
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in merge_keys])

    # Caso 1: tabela ainda não existe - cria a tabela
    if not spark.catalog.tableExists(table_name):
        (df_source.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ano_edicao")
            .option("userMetadata", description)
            .saveAsTable(table_name))
        print(f"Criada: {table_name} - ano {ano_edicao}")
        return

    deltaTable = DeltaTable.forName(spark, table_name)

    # Caso 2: tabela existente sem ano_edicao - realiza a migração
    # Armazena o DataFrame uma vez para evitar RPCs de Analyze repetidos no Spark Connect
    target_df = deltaTable.toDF()
    target_cols = target_df.columns
    if "ano_edicao" not in target_cols:
        print(f"Migrando {table_name}: adicionando a coluna de partição ano_edicao...")
        if "reference_year" in target_cols:
            # Retroalimenta ano_edicao a partir da coluna reference_year existente
            existing_df = target_df.withColumns(
                {"ano_edicao": F.col("reference_year").cast("int")}
            )
        else:
            # Sem coluna reference_year (ex.: tabela de dicionário) - usa o valor do parâmetro
            existing_df = target_df.withColumns(
                {"ano_edicao": F.lit(ano_edicao)}
            )
        # Posiciona ano_edicao em segundo — após a chave da entidade, não como metadado no final
        first_col = target_cols[0]
        rest_cols = target_cols[1:]
        existing_df = existing_df.select(first_col, "ano_edicao", *rest_cols)
        (existing_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ano_edicao")
            .option("userMetadata", "Migracao: coluna de particao ano_edicao adicionada")
            .saveAsTable(table_name))
        deltaTable = DeltaTable.forName(spark, table_name)
        target_cols = [first_col, "ano_edicao", *rest_cols]
    else:
        # Garante que ano_edicao fique em segundo (após a chave da entidade de merge_keys)
        entity_key = next(k for k in merge_keys if k != "ano_edicao")
        if len(target_cols) < 2 or target_cols[0] != entity_key or target_cols[1] != "ano_edicao":
            # Move ano_edicao para o início e depois entity_key para o início (resultado: entity_key, ano_edicao, ...)
            spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN ano_edicao FIRST")
            spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {entity_key} FIRST")
            deltaTable = DeltaTable.forName(spark, table_name)
            target_cols = [entity_key, "ano_edicao"] + [c for c in target_cols if c not in (entity_key, "ano_edicao")]

    # Evolução de schema: adiciona as colunas novas da origem ausentes no destino
    missing_cols = [c for c in source_cols if c not in target_cols]
    if missing_cols:
        for col in missing_cols:
            spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {col} STRING")
        print(f"Colunas adicionadas a {table_name}: {missing_cols}")
        deltaTable = DeltaTable.forName(spark, table_name)

    # MERGE: atualiza as linhas correspondentes e insere as novas
    (deltaTable.alias("target")
        .merge(df_source.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"MERGE concluído: {table_name} - ano {ano_edicao}")


